# LeetCode 2451 — Odd String Difference bằng Python

Link đề: https://leetcode.com/problems/odd-string-difference/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, String

Skeleton LeetCode:

```python
class Solution:
    def oddString(self, words: List[str]) -> str:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu **mảng hiệu (difference array)** của một chuỗi: hiệu giữa hai chữ cái liền kề theo vị trí trong bảng chữ cái, và vì sao nó **không đổi** khi dịch cả chuỗi sang phải/trái.
2. Dùng `ord`, `zip`, `tuple` để tính mảng hiệu và dùng `tuple` làm **khóa của dict** (vì `list` không băm được).
3. Biết kỹ thuật **nhóm theo khóa chuẩn hóa** (canonical key) rồi tìm nhóm chỉ có một phần tử.
4. Viết 4 cách: so sánh từng cặp từ (brute), nhóm bằng `dict`, `Counter` của khóa, và so ba từ đầu để biết "mẫu chung" (không cần cấu trúc phụ); đối chiếu bằng test ngẫu nhiên.
5. Tránh các lỗi: chỉ xét hiệu đầu tiên, dùng tổng hiệu (hiệu đầu-cuối), bỏ dấu (`abs`), và so sánh cả từ thay vì mảng hiệu.

## 2. Đề bài & ví dụ

Mỗi chuỗi `words[i]` có độ dài `n` (cùng độ dài). Với một chuỗi, **mảng hiệu** `difference[j] = words[i][j+1] - words[i][j]` với `0 <= j <= n - 2`, trong đó mỗi chữ cái được đánh số theo vị trí trong bảng chữ cái: `'a' = 0`, `'b' = 1`, ..., `'z' = 25`. Hiệu có thể **âm**.

Ví dụ `"acb"`: `difference = [2 - 0, 1 - 2] = [2, -1]`.

Tất cả chuỗi trong `words` có cùng mảng hiệu **trừ đúng một chuỗi** có mảng hiệu khác. Trả về **chuỗi có mảng hiệu khác** đó.

| Ví dụ | words | Output | Giải thích |
|---|---|---|---|
| 1 | `["adc","wzy","abc"]` | `"abc"` | `adc` → `[3,-1]`; `wzy` → `[3,-1]`; `abc` → `[1,1]` |
| 2 | `["aaa","bob","ccc","ddd"]` | `"bob"` | `aaa`, `ccc`, `ddd` → `[0,0]`; `bob` → `[13,-13]` |

Constraints: `3 <= words.length <= 100`; `n == words[i].length`; `2 <= n <= 20`; `words[i]` gồm chữ cái thường.

In [1]:
def diff(word):
    return [ord(b) - ord(a) for a, b in zip(word, word[1:])]

for w in ("adc", "wzy", "abc", "acb"):
    print(w, diff(w))

adc [3, -1]
wzy [3, -1]
abc [1, 1]
acb [2, -1]


## 3. Kiến thức nền

### 3.1. `ord` và vị trí chữ cái

`ord('a') = 97`. Vị trí trong bảng chữ cái là `ord(c) - ord('a')`. Với **hiệu** của hai chữ cái, bù trừ `ord('a')` triệt tiêu, nên chỉ cần `ord(b) - ord(a)`.

In [2]:
print(ord('a'), ord('b'), ord('z'))
print(ord('d') - ord('a'), ord('c') - ord('d'))      # 3, -1

97 98 122
3 -1


### 3.2. `zip(word, word[1:])` để lấy cặp chữ cái liền kề

`word[1:]` dịch chuỗi sang trái một bước; `zip(word, word[1:])` cho cặp `(word[j], word[j+1])`. Với `n` chữ cái ta có `n - 1` cặp, đúng độ dài mảng hiệu.

In [3]:
word = "adc"
print(list(zip(word, word[1:])))
print([ord(b) - ord(a) for a, b in zip(word, word[1:])])

[('a', 'd'), ('d', 'c')]
[3, -1]


### 3.3. Mảng hiệu bất biến khi dịch chuỗi

Nếu cộng cùng một số vào mọi chữ cái (dịch chuỗi), các hiệu liền kề **không đổi**. Đó là lý do `"adc"` và `"wzy"` (dịch `+22`) có cùng mảng hiệu `[3, -1]`. Nói cách khác, mảng hiệu là một **dạng chuẩn hóa** (canonical form) của "hình dạng" chuỗi, bỏ qua điểm bắt đầu.

In [4]:
def diff(word):
    return [ord(b) - ord(a) for a, b in zip(word, word[1:])]

print(diff("adc"), diff("wzy"), diff("beb"))

[3, -1] [3, -1] [3, -3]


### 3.4. `tuple` làm khóa của `dict`

`list` là đối tượng **có thể thay đổi** nên không băm được; `d[[1, 2]]` gây `TypeError`. Dùng `tuple([1, 2])` (bất biến, băm được) làm khóa.

In [5]:
groups = {}
for w in ["adc", "wzy", "abc"]:
    key = tuple(ord(b) - ord(a) for a, b in zip(w, w[1:]))
    groups.setdefault(key, []).append(w)
print(groups)
try:
    {}[[1, 2]] = 0
except TypeError as e:
    print("list không băm được:", e)

{(3, -1): ['adc', 'wzy'], (1, 1): ['abc']}
list không băm được: unhashable type: 'list'


### 3.5. `setdefault` và `Counter`

`d.setdefault(k, []).append(x)` gom các phần tử cùng khóa vào cùng danh sách. `Counter(keys)` đếm số lần xuất hiện của mỗi khóa.

In [6]:
from collections import Counter
keys = [(3, -1), (3, -1), (1, 1)]
print(Counter(keys))
print([k for k, c in Counter(keys).items() if c == 1])

Counter({(3, -1): 2, (1, 1): 1})
[(1, 1)]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Tính tay ví dụ 1: `["adc", "wzy", "abc"]`.

| Từ | Vị trí | Hiệu |
|---|---|---|
| `adc` | `0, 3, 2` | `3, -1` |
| `wzy` | `22, 25, 24` | `3, -1` |
| `abc` | `0, 1, 2` | `1, 1` |

Hai từ đầu có cùng mảng hiệu `[3, -1]` còn `abc` có `[1, 1]` ⇒ `abc` là từ "lạ".

Ý tưởng chung: **tính mảng hiệu của mọi từ, rồi tìm từ có mảng hiệu duy nhất**. Vì chỉ có đúng một từ lạ và các từ còn lại (ít nhất 2) cùng mảng hiệu, từ có mảng hiệu xuất hiện **đúng một lần** chính là đáp án.

In [7]:
words = ["aaa", "bob", "ccc", "ddd"]
for w in words:
    print(f"{w}: {[ord(b) - ord(a) for a, b in zip(w, w[1:])]}")

aaa: [0, 0]
bob: [13, -13]
ccc: [0, 0]
ddd: [0, 0]


## 5. Cách 1 — Brute Force / baseline: so sánh từng từ với mọi từ khác

**Ý tưởng:** với mỗi từ `w_i`, so sánh mảng hiệu với mỗi từ khác `w_j` (`j != i`). Từ nào **không** trùng với bất kỳ từ khác nào là từ lạ. Vì có ít nhất 2 từ "bình thường" cùng mảng hiệu, mỗi từ bình thường luôn có ít nhất một bạn trùng.

In [8]:
from typing import List

def diff_of(word):
    return [ord(b) - ord(a) for a, b in zip(word, word[1:])]

class SolutionBrute:
    def oddString(self, words: List[str]) -> str:
        for i in range(len(words)):
            matched = False
            for j in range(len(words)):
                if i != j and diff_of(words[i]) == diff_of(words[j]):
                    matched = True
                    break
            if not matched:
                return words[i]
        return ''

In [9]:
sol = SolutionBrute()
print(sol.oddString(["adc", "wzy", "abc"]))            # abc
print(sol.oddString(["aaa", "bob", "ccc", "ddd"]))     # bob

abc
bob


**Độ phức tạp:** `m = len(words)`, `n = len(words[i])`: `O(m^2 · n)` (mỗi lần so sánh tính lại hiệu `O(n)`). Với `m = 100`, `n = 20` khoảng `2 × 10^5` thao tác, vẫn nhanh nhưng lãng phí. Space `O(n)`. **Nút thắt:** tính lại cùng mảng hiệu nhiều lần và so sánh từng cặp; nên tính mỗi mảng hiệu **một lần** và nhóm lại.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** dùng mảng hiệu (đã chuyển thành `tuple`) làm **khóa chuẩn hóa**; các từ có cùng "hình dạng" rơi vào cùng một nhóm.

- **Invariant:** sau khi xử lý `k` từ đầu, `groups[key]` chứa đúng các từ trong `k` từ đó có mảng hiệu `key`.
- **Quy tắc kết luận:** sau khi nhóm xong, **nhóm có đúng 1 từ** là nhóm của từ lạ (vì mọi từ còn lại, ít nhất 2 từ, dồn vào một nhóm lớn).
- Cách nhẹ hơn không cần dict: lấy ba từ đầu; hai từ trong ba từ đó chắc chắn cùng mảng hiệu (vì chỉ một từ lạ). Từ đó suy ra mảng hiệu "chuẩn" và quét tìm từ khác mảng chuẩn.

## 7. Các cách tiếp theo

### Cách 2 — Nhóm bằng `dict` rồi lấy nhóm có một phần tử

In [10]:
from typing import List

class SolutionGroup:
    def oddString(self, words: List[str]) -> str:
        groups = {}
        for w in words:
            key = tuple(ord(b) - ord(a) for a, b in zip(w, w[1:]))
            groups.setdefault(key, []).append(w)
        for lst in groups.values():
            if len(lst) == 1:
                return lst[0]
        return ''

In [11]:
print(SolutionGroup().oddString(["adc", "wzy", "abc"]))            # abc
print(SolutionGroup().oddString(["aaa", "bob", "ccc", "ddd"]))     # bob

abc
bob


**Độ phức tạp:** Time `O(m · n)` (mỗi từ tính hiệu `O(n)` và băm `O(n)`), Space `O(m · n)` cho các nhóm.

### Cách 3 — `Counter` của khóa, rồi tìm từ có khóa đếm bằng 1

Không cần lưu danh sách từ trong nhóm: chỉ cần đếm số lần mỗi khóa, sau đó duyệt lại để tìm từ có khóa đếm `1`.

In [12]:
from typing import List
from collections import Counter

class SolutionCounter:
    def oddString(self, words: List[str]) -> str:
        keys = [tuple(ord(b) - ord(a) for a, b in zip(w, w[1:])) for w in words]
        cnt = Counter(keys)
        for w, k in zip(words, keys):
            if cnt[k] == 1:
                return w
        return ''

In [13]:
print(SolutionCounter().oddString(["adc", "wzy", "abc"]))            # abc
print(SolutionCounter().oddString(["aaa", "bob", "ccc", "ddd"]))     # bob

abc
bob


**Độ phức tạp:** Time `O(m · n)`, Space `O(m · n)` cho danh sách khóa (có thể giảm xuống số khóa phân biệt, tối đa 2).

### Cách 4 — So ba từ đầu để biết mẫu chung

Vì chỉ có một từ lạ, trong `words[0], words[1], words[2]` ít nhất hai từ cùng mảng hiệu.

- Nếu `diff(w0) == diff(w1)`: mẫu chung là `diff(w0)`; từ lạ nằm ở `w2` trở đi ⇒ quét tìm từ có mảng hiệu khác mẫu (`w2` cũng được kiểm tra).
- Nếu `diff(w0) != diff(w1)`: một trong hai là từ lạ; `w2` quyết định: nếu `diff(w2) == diff(w0)` thì lạ là `w1`, ngược lại là `w0`.

In [14]:
from typing import List

class SolutionThree:
    def oddString(self, words: List[str]) -> str:
        def diff(w):
            return tuple(ord(b) - ord(a) for a, b in zip(w, w[1:]))

        d0, d1, d2 = diff(words[0]), diff(words[1]), diff(words[2])
        if d0 != d1:
            return words[1] if d2 == d0 else words[0]
        for w in words[2:]:
            if diff(w) != d0:
                return w
        return ''

In [15]:
print(SolutionThree().oddString(["adc", "wzy", "abc"]))            # abc
print(SolutionThree().oddString(["aaa", "bob", "ccc", "ddd"]))     # bob
print(SolutionThree().oddString(["bob", "aaa", "ccc", "ddd"]))     # bob (từ lạ ở đầu)

abc
bob
bob


**Độ phức tạp:** Time `O(m · n)` trong trường hợp xấu nhất (quét hết) nhưng thường dừng sớm, Space `O(n)`. Cách này không lưu cấu trúc phụ theo số từ.

## 8. TRACE từng bước

Cách `dict` trên `["aaa", "bob", "ccc", "ddd"]`:

| Bước | `w` | `key` | `groups` sau |
|---|---|---|---|
| 1 | `aaa` | `(0, 0)` | `{(0,0): [aaa]}` |
| 2 | `bob` | `(13, -13)` | `{(0,0): [aaa], (13,-13): [bob]}` |
| 3 | `ccc` | `(0, 0)` | `{(0,0): [aaa, ccc], (13,-13): [bob]}` |
| 4 | `ddd` | `(0, 0)` | `{(0,0): [aaa, ccc, ddd], (13,-13): [bob]}` |

Nhóm có đúng một từ là `(13, -13): [bob]` ⇒ đáp án `bob`.

In [16]:
def trace_group(words):
    groups = {}
    for step, w in enumerate(words, 1):
        key = tuple(ord(b) - ord(a) for a, b in zip(w, w[1:]))
        groups.setdefault(key, []).append(w)
        print(f"bước {step}: w={w!r}, key={key}, groups={groups}")
    for key, lst in groups.items():
        if len(lst) == 1:
            print("nhóm có 1 từ:", key, "->", lst[0])
            return lst[0]

print(trace_group(["aaa", "bob", "ccc", "ddd"]))

bước 1: w='aaa', key=(0, 0), groups={(0, 0): ['aaa']}
bước 2: w='bob', key=(13, -13), groups={(0, 0): ['aaa'], (13, -13): ['bob']}
bước 3: w='ccc', key=(0, 0), groups={(0, 0): ['aaa', 'ccc'], (13, -13): ['bob']}
bước 4: w='ddd', key=(0, 0), groups={(0, 0): ['aaa', 'ccc', 'ddd'], (13, -13): ['bob']}
nhóm có 1 từ: (13, -13) -> bob
bob


## 9. Code nộp LeetCode

Dùng cách `dict` nhóm (rõ ràng và an toàn):

In [17]:
from typing import List

class Solution:
    def oddString(self, words: List[str]) -> str:
        groups = {}
        for w in words:
            key = tuple(ord(b) - ord(a) for a, b in zip(w, w[1:]))
            groups.setdefault(key, []).append(w)
        for lst in groups.values():
            if len(lst) == 1:
                return lst[0]
        return ''

In [18]:
print(Solution().oddString(["adc", "wzy", "abc"]))
print(Solution().oddString(["aaa", "bob", "ccc", "ddd"]))

abc
bob


## 10. Vì sao thuật toán đúng

1. **Mảng hiệu là hàm của từ.** Hai từ có cùng mảng hiệu thì thuộc cùng nhóm; từ nào có mảng hiệu khác thì thuộc nhóm khác (khóa `tuple` so sánh bằng từng phần tử).
2. **Cấu trúc nhóm.** Đề đảm bảo mọi từ trừ một từ cùng mảng hiệu `D`, và từ còn lại có mảng hiệu `D' != D`. Vì `m >= 3`, nhóm `D` có `m - 1 >= 2` từ, còn nhóm `D'` đúng `1` từ.
3. **Tìm nhóm.** Nhóm có độ dài `1` là nhóm duy nhất có đúng một từ (nhóm kia có `>= 2`), nên từ đó là đáp án; không thể nhầm.
4. **Cách đếm** (Counter) làm đúng điều tương tự: từ có khóa đếm `1`.
5. **Cách ba từ đầu.** Trong ba từ `w0, w1, w2` có nhiều nhất một từ lạ, nên ít nhất hai từ cùng mảng `D`. Nếu `d0 == d1` thì `D = d0`; từ lạ khác `D` nên nằm ở `w2..` và quét tìm được. Nếu `d0 != d1` thì một trong hai là từ lạ, còn `w2` (chắc chắn bình thường vì chỉ có một từ lạ) có mảng `D` bằng mảng của từ bình thường trong `{w0, w1}`: so `d2` với `d0` để biết `w0` là bình thường (khi đó từ lạ là `w1`) hay không (khi đó từ lạ là `w0`).
6. **Brute.** Từ bình thường luôn có ít nhất một "bạn" cùng mảng hiệu; từ lạ không có bạn nào. Nên từ không có bạn chính là từ lạ.

## 11. Phân tích Time / Space Complexity

Đặt `m = len(words)`, `n = len(words[i])`.

| Cách | Time | Space |
|---|---|---|
| Brute (so từng cặp) | `O(m^2 · n)` | `O(n)` |
| `dict` nhóm | `O(m · n)` | `O(m · n)` |
| `Counter` khóa | `O(m · n)` | `O(m · n)` |
| So ba từ đầu | `O(m · n)` xấu nhất | `O(n)` |

Ràng buộc `m <= 100`, `n <= 20` làm mọi cách chạy tức thì. Cách `dict` và `Counter` là chuẩn; cách ba từ đầu thể hiện tận dụng "chỉ một từ lạ".

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

In [19]:
sols = [SolutionBrute(), SolutionGroup(), SolutionCounter(), SolutionThree(), Solution()]
cases = [
    (["adc", "wzy", "abc"], "abc"),                     # ví dụ 1
    (["aaa", "bob", "ccc", "ddd"], "bob"),              # ví dụ 2
    (["abc", "bcd", "abd"], "abd"),                     # cùng hiệu đầu, khác hiệu sau
    (["ace", "bdf", "ade"], "ade"),                     # cùng tổng hiệu (đầu-cuối), khác mảng
    (["abc", "bcd", "aba"], "aba"),                     # khác nhau chỉ ở dấu (1,-1 vs 1,1)
    (["bob", "aaa", "ccc", "ddd"], "bob"),              # từ lạ ở đầu
    (["aaa", "ccc", "bob", "ddd"], "bob"),              # từ lạ ở giữa
    (["aaa", "ccc", "ddd", "bob"], "bob"),              # từ lạ ở cuối
    (["za", "yz", "ab"], "za"),                         # n = 2: za -> -25; yz -> 1; ab -> 1
    (["ab", "bc", "za"], "za"),                         # n = 2, hiệu âm -25 vs 1
    (["az", "ab", "bc"], "az"),                         # n = 2, hiệu 25 vs 1
    (["zzzz", "aaaa", "abab"], "abab"),                 # dài 4
]
for words, want in cases:
    for s in sols:
        assert s.oddString(words) == want, (type(s).__name__, words, s.oddString(words), want)
print("tất cả test cố định đều đúng với cả 5 cách:", len(cases))

tất cả test cố định đều đúng với cả 5 cách: 12


### 12.2. Đối chiếu ngẫu nhiên (có seed)

Sinh dữ liệu **đúng đề**: chọn mảng hiệu mẫu `D` và mảng hiệu khác `D'`; tạo `m - 1` từ có hiệu `D` với điểm bắt đầu khác nhau (đảm bảo mọi chữ cái trong `a..z`), và một từ có hiệu `D'`; xáo trộn vị trí. Có hàm kiểm tra ràng buộc.

In [20]:
import random
import string
random.seed(2451)

def random_pattern(n):
    # chọn mảng hiệu độ dài n-1 sao cho dải giá trị (max-min của tổng tiền tố) <= 25
    while True:
        d = [random.randint(-6, 6) for _ in range(n - 1)]
        pref = [0]
        for x in d:
            pref.append(pref[-1] + x)
        if max(pref) - min(pref) <= 25:
            return d, pref

def build_word(pref):
    lo = -min(pref)
    hi = 25 - max(pref)
    start = random.randint(lo, hi)
    return "".join(chr(ord('a') + start + p) for p in pref)

def diff_t(w):
    return tuple(ord(b) - ord(a) for a, b in zip(w, w[1:]))

def gen():
    n = random.randint(2, 20)
    m = random.randint(3, 100)
    d, pref = random_pattern(n)
    while True:
        d2, pref2 = random_pattern(n)
        if d2 != d:
            break
    normal = [build_word(pref) for _ in range(m - 1)]
    odd = build_word(pref2)
    words = normal + [odd]
    random.shuffle(words)
    return words, odd

def check(words):
    assert 3 <= len(words) <= 100
    n = len(words[0])
    assert 2 <= n <= 20 and all(len(w) == n and w.islower() and w.isalpha() for w in words)
    from collections import Counter
    c = Counter(diff_t(w) for w in words)
    assert sorted(c.values()) == [1, len(words) - 1]       # đúng một từ lạ

for _ in range(1000):
    words, odd = gen()
    check(words)
    for s in sols:
        assert s.oddString(words) == odd, (type(s).__name__, words, odd)
print("1000 test ngẫu nhiên đúng ràng buộc: 5 cách đều tìm đúng từ lạ")

1000 test ngẫu nhiên đúng ràng buộc: 5 cách đều tìm đúng từ lạ


### 12.3. Test kích thước lớn nhất và nhỏ nhất

Kiểm tra `m = 100`, `n = 20` và `m = 3`, `n = 2`. Với từ lạ ở mỗi vị trí có thể, đảm bảo cách ba từ đầu (có nhánh riêng) cũng đúng.

In [21]:
random.seed(5)
# m = 3, n = 2: thử từ lạ ở cả 3 vị trí
for _ in range(300):
    d, pref = random_pattern(2)
    while True:
        d2, pref2 = random_pattern(2)
        if d2 != d:
            break
    a, b, odd = build_word(pref), build_word(pref), build_word(pref2)
    for pos in range(3):
        words = [a, b]
        words.insert(pos, odd)
        check(words)
        for s in sols:
            assert s.oddString(words) == odd, (type(s).__name__, words)

# m = 100, n = 20
for _ in range(20):
    d, pref = random_pattern(20)
    while True:
        d2, pref2 = random_pattern(20)
        if d2 != d:
            break
    words = [build_word(pref) for _ in range(99)]
    odd = build_word(pref2)
    words.insert(random.randint(0, 99), odd)
    check(words)
    for s in sols:
        assert s.oddString(words) == odd
print("test kích thước biên: OK")

test kích thước biên: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ so sánh hiệu đầu tiên

In [22]:
from typing import List

def sai_chi_hieu_dau(words: List[str]) -> str:
    groups = {}
    for w in words:
        key = ord(w[1]) - ord(w[0])                       # SAI: chỉ hiệu đầu tiên
        groups.setdefault(key, []).append(w)
    for lst in groups.values():
        if len(lst) == 1:
            return lst[0]
    return ""

words = ["abc", "bcd", "abd"]
print("đúng:", repr(Solution().oddString(words)), "| sai:", repr(sai_chi_hieu_dau(words)))
assert Solution().oddString(words) != sai_chi_hieu_dau(words)

đúng: 'abd' | sai: ''


**Vì sao sai:** mảng hiệu gồm **cả** `n - 1` phần tử. `"abd"` có hiệu `[1, 2]` khác `[1, 1]` ở phần tử thứ hai, nhưng hiệu đầu đều là `1` nên code sai không tách được và trả chuỗi rỗng.

### Lỗi 2: Dùng tổng hiệu (chữ cuối trừ chữ đầu)

In [23]:
def sai_tong_hieu(words: List[str]) -> str:
    groups = {}
    for w in words:
        key = ord(w[-1]) - ord(w[0])                       # SAI: tổng các hiệu = hiệu đầu-cuối
        groups.setdefault(key, []).append(w)
    for lst in groups.values():
        if len(lst) == 1:
            return lst[0]
    return ""

words = ["ace", "bdf", "ade"]
print("đúng:", repr(Solution().oddString(words)), "| sai:", repr(sai_tong_hieu(words)))
assert Solution().oddString(words) != sai_tong_hieu(words)

đúng: 'ade' | sai: ''


**Vì sao sai:** tổng các hiệu liền kề luôn bằng chữ cuối trừ chữ đầu; hai mảng hiệu khác nhau có thể có cùng tổng (`[2,2]` và `[3,1]` cùng tổng `4`). Phải so sánh **toàn bộ mảng**.

### Lỗi 3: Bỏ dấu của hiệu (`abs`)

In [24]:
def sai_abs(words: List[str]) -> str:
    groups = {}
    for w in words:
        key = tuple(abs(ord(b) - ord(a)) for a, b in zip(w, w[1:]))      # SAI: abs làm mất dấu
        groups.setdefault(key, []).append(w)
    for lst in groups.values():
        if len(lst) == 1:
            return lst[0]
    return ""

words = ["abc", "bcd", "aba"]
print("đúng:", repr(Solution().oddString(words)), "| sai:", repr(sai_abs(words)))
assert Solution().oddString(words) != sai_abs(words)

đúng: 'aba' | sai: ''


**Vì sao sai:** `"abc"` có `[1, 1]` còn `"aba"` có `[1, -1]`. Dấu phân biệt "đi lên" và "đi xuống"; `abs` làm cả hai thành `[1, 1]`.

### Lỗi 4: So sánh cả từ thay vì mảng hiệu

In [25]:
from collections import Counter

def sai_so_ca_tu(words: List[str]) -> str:
    cnt = Counter(words)                                   # SAI: đếm từ, không phải mảng hiệu
    for w in words:
        if cnt[w] == 1:
            return w
    return ""

words = ["adc", "wzy", "abc"]
print("đúng:", repr(Solution().oddString(words)), "| sai:", repr(sai_so_ca_tu(words)))
assert Solution().oddString(words) != sai_so_ca_tu(words)

đúng: 'abc' | sai: 'adc'


**Vì sao sai:** ba từ đều khác nhau nên đều "xuất hiện một lần"; code sai trả từ đầu tiên. Đề nhóm theo **hình dạng** (mảng hiệu), không theo nội dung từ.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tất cả giống nhau trừ một" ⇒ **chuẩn hóa mỗi phần tử về một khóa** rồi tìm khóa có tần suất `1`.
- Khi các đối tượng bằng nhau "sau khi dịch" ⇒ dùng **mảng hiệu** (difference) làm khóa; tương tự bài nhóm chuỗi dịch (Group Shifted Strings).
- Mảng hiệu cần được chuyển thành `tuple` (hoặc chuỗi) để làm khóa `dict`.
- Với `Counter`, đừng quên **duyệt lại `words`** để lấy đúng từ (khóa không cho biết từ nào).
- Tận dụng "chỉ một từ lạ": ba từ đầu đủ để biết mẫu chung; tuy nhiên cách nhóm vẫn dễ chứng minh hơn.
- Thử các test: từ lạ ở đầu / giữa / cuối; `n = 2`; hiệu âm.

## 15. Biến thể của bài

1. **Group Shifted Strings:** nhóm mọi chuỗi cùng hình dạng (có thể quay vòng `z → a`: dùng `% 26` cho hiệu).
2. **Nhiều từ lạ:** trả về tất cả các từ có mảng hiệu xuất hiện ít nhất.
3. **Trả về chỉ số** của từ lạ thay vì từ.
4. **Chuỗi độ dài khác nhau:** thêm độ dài vào khóa.
5. **Hiệu theo vòng** (mod 26): hai từ bằng nhau nếu hiệu bằng nhau theo modulo 26.
6. **Mảng số** thay vì chuỗi: cùng ý tưởng với `nums[i+1] - nums[i]`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute | so mảng hiệu từng từ với mọi từ khác | `O(m^2 · n)` | `O(n)` | `m` rất nhỏ, đối chứng |
| 2. `dict` nhóm | khóa là `tuple` hiệu, lấy nhóm có 1 từ | `O(m · n)` | `O(m · n)` | cách chuẩn, dễ chứng minh |
| 3. `Counter` | đếm khóa rồi tìm từ có khóa đếm 1 | `O(m · n)` | `O(m · n)` | ngắn gọn, ít cấu trúc |
| 4. Ba từ đầu | suy mẫu chung từ ba từ, quét từ khác mẫu | `O(m · n)` | `O(n)` | tận dụng "chỉ một từ lạ", ít bộ nhớ |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Mảng hiệu của `"bdb"` là gì? Còn của `"zzz"`?

**Bài 2 (tính tay).** Với `["ace", "bdf", "xyz"]`, hãy tính mảng hiệu cả ba (chú ý `xyz`), rồi nói từ nào "lạ" nếu đề đảm bảo chỉ một từ lạ.

**Bài 3 (điền TODO).** Hoàn thành hàm `khoa(word)` trả về `tuple` mảng hiệu.

In [26]:
def khoa(word):
    # TODO: trả về tuple các hiệu ord(word[j+1]) - ord(word[j])
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `nhom_dich(words)` trả về danh sách các nhóm (mỗi nhóm là danh sách từ có cùng mảng hiệu), giữ thứ tự xuất hiện.

**Bài 5 (biến thể / mini challenge).** Cho phép quay vòng bảng chữ cái: hiệu tính theo modulo 26 (ví dụ `"az"` và `"ba"` cùng hình dạng vì `z - a = 25` và `a - b = -1 ≡ 25`). Viết `nhom_dich_vong(words)` và chứng minh bằng ví dụ.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `b→d = 2`, `d→b = -2`; `zzz` cho `[0, 0]`.
- **Mức 2 (vừa):** Bài 2: `ace = [2, 2]`, `bdf = [2, 2]`, `xyz = [1, 1]`. Bài 3: `tuple(ord(b) - ord(a) for a, b in zip(word, word[1:]))`.
- **Mức 3 (sâu):** Bài 4: dict `setdefault(key, []).append(w)` rồi `list(groups.values())` (dict giữ thứ tự chèn). Bài 5: dùng khóa `(ord(b) - ord(a)) % 26`; Python cho kết quả không âm với `%` và số nguyên dương modul.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [27]:
# Bài 1
def khoa(word):
    return tuple(ord(b) - ord(a) for a, b in zip(word, word[1:]))

assert khoa("bdb") == (2, -2)
assert khoa("zzz") == (0, 0)

# Bài 2: xyz = (1, 1) là từ lạ
assert khoa("ace") == (2, 2) and khoa("bdf") == (2, 2) and khoa("xyz") == (1, 1)
assert Solution().oddString(["ace", "bdf", "xyz"]) == "xyz"

# Bài 4
def nhom_dich(words):
    groups = {}
    for w in words:
        groups.setdefault(khoa(w), []).append(w)
    return list(groups.values())

# Bài 5
def nhom_dich_vong(words):
    groups = {}
    for w in words:
        key = tuple((ord(b) - ord(a)) % 26 for a, b in zip(w, w[1:]))
        groups.setdefault(key, []).append(w)
    return list(groups.values())

assert nhom_dich(["abc", "bcd", "ace", "xyz"]) == [["abc", "bcd", "xyz"], ["ace"]]
assert nhom_dich(["a"]) == [["a"]]
assert nhom_dich(["aa", "bb", "ab"]) == [["aa", "bb"], ["ab"]]
assert nhom_dich_vong(["az", "ba"]) == [["az", "ba"]]
assert nhom_dich(["az", "ba"]) == [["az"], ["ba"]]            # không quay vòng thì tách riêng
assert nhom_dich_vong(["abc", "xyz", "ace"]) == [["abc", "xyz"], ["ace"]]
assert khoa("a") == ()
print("tất cả bài tập OK")

tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (nhóm theo khóa chuẩn hóa)

```python
class Solution:
    def oddString(self, words: List[str]) -> str:
        groups = {}
        for w in words:
            key = tuple(ord(b) - ord(a) for a, b in zip(w, w[1:]))
            groups.setdefault(key, []).append(w)
        for lst in groups.values():
            if len(lst) == 1:
                return lst[0]
        return ""
```

**Checklist**

- [ ] Khóa là **toàn bộ** mảng hiệu (`tuple`), không phải một phần tử hoặc tổng?
- [ ] Giữ dấu của hiệu (không dùng `abs`)?
- [ ] Dùng `tuple` (không phải `list`) làm khóa `dict`?
- [ ] Trả về **từ** (chuỗi), không phải khóa hay chỉ số?
- [ ] Đã thử từ lạ ở đầu, giữa, cuối và trường hợp `n = 2`?
- [ ] Nếu dùng `Counter`: đã duyệt lại `words` để tìm từ có khóa đếm `1`?
- [ ] Signature đúng `oddString(self, words: List[str]) -> str` và có `from typing import List`?